Loading the Breast Cancer Wisconsin dataset from the sklearn library.

In [1]:
#Import all necessary libraries

from sklearn.datasets import load_breast_cancer
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# Load dataset
data = load_breast_cancer()

# Create DataFrame for features
X = pd.DataFrame(data.data, columns=data.feature_names)

# Create Series for target
y = pd.Series(data.target, name="target")

# Map target labels to class names
target_names = {i: name for i, name in enumerate(data.target_names)}
y_named = y.map(target_names)

print("======================================================================")
print("Display first rows")
print(y_named.head())

# Combine numeric target
df_numeric = pd.concat([X, y], axis=1)

# Combine named target (more interpretable)
df_named = pd.concat([X, y_named.rename("diagnosis")], axis=1)

print("======================================================================")
print("Display first rows mapped")
print(df_named.head())

# Compute target class distribution

# Count samples per class
class_counts = y_named.value_counts()

# Percentage distribution
class_percentages = y_named.value_counts(normalize=True) * 100

class_distribution = pd.DataFrame({
    "Count": class_counts,
    "Percentage (%)": class_percentages.round(2)
})

print("======================================================================")
print("Class Distribution")
print(class_distribution)

# Identify class imbalance
imbalance_ratio = class_counts.max() / class_counts.min()
print("======================================================================")
print("Class Imbalance Ratio")
print(imbalance_ratio)

Display first rows
0    malignant
1    malignant
2    malignant
3    malignant
4    malignant
Name: target, dtype: object
Display first rows mapped
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280 

Check for missing values and duplicate rows in the dataset.

In [2]:
# Display missing values per column
print("======================================================================")
print("Display missing values per column")
missing_values = df_numeric.isnull().sum().sum()
if missing_values == 0:
    print("No missing values found in the dataset.")
else:
    print(f"Found {missing_values} missing value(s) in the dataset.")

Display missing values per column
No missing values found in the dataset.


In [3]:
# Display whether dublicated rows exist
print("======================================================================")
print("Display whether dublicated rows exist")
num_duplicates = df_numeric.duplicated().sum()

if num_duplicates == 0:
    print("No duplicate rows found in the dataset.")
else:
    print(f"Found {num_duplicates} duplicate rows in the dataset.")

Display whether dublicated rows exist
No duplicate rows found in the dataset.


Calculate standard deviation for each column and display top 5

In [4]:
std_dev = df_numeric.std()
top_5_std = std_dev.sort_values(ascending=False).head(5)
print("======================================================================")
print("Display display top 5 Standard Deviation attributes")
print(top_5_std)

Display display top 5 Standard Deviation attributes
worst area         569.356993
mean area          351.914129
area error          45.491006
worst perimeter     33.602542
mean perimeter      24.298981
dtype: float64


Split data in Train and Test

In [5]:
# Split the dataset in stratified Train/Test in 80/20 proportions
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Cunstruct pipeline for Robust Scaler Decision Tree
dt_with_scaler = Pipeline([
    ("scaler", RobustScaler()),
    ("dt", DecisionTreeClassifier(max_depth=3, random_state=42))
])

# Fit the Model
dt_with_scaler.fit(X_train, y_train)

# Calculate Predictions
y_pred_with_scaler = dt_with_scaler.predict(X_test)
acc_with_scaler = accuracy_score(y_test, y_pred_with_scaler)

# Cunstruct pipeline for Robust Scaler Decision Tree
dt_no_scaler = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

# Fit the Model
dt_no_scaler.fit(X_train, y_train)

# Calculate Predictions
y_pred_no_scaler = dt_no_scaler.predict(X_test)
acc_no_scaler = accuracy_score(y_test, y_pred_no_scaler)

# Display Accuracy for both Models
print("Accuracy with RobustScaler:", acc_with_scaler)
print("Accuracy without RobustScaler:", acc_no_scaler)

Accuracy with RobustScaler: 0.9385964912280702
Accuracy without RobustScaler: 0.9385964912280702


Calculation for the two cases provides almost identical accuracy. This is expected since RobustScaler is scaling the data, but scaling does not provide any aditional benefit on the Desicion Tree models. This explained because desicion trees does not compare distances or gradients, only compaire values.

Export Structure of Model (No RobustScaler), identify the root node’s splitting feature
and the total number of leaves.

In [ ]:
print("======================================================================")
print("Model's Structure")
feature_names = X_train.columns # or a list of feature names
tree_text = export_text(dt_no_scaler, feature_names=list(feature_names))
print(tree_text)
print("======================================================================")
print("Identify root node’s Splitting Feature")
tree = dt_no_scaler.tree_
root_feature_index = tree.feature[0]
root_threshold = tree.threshold[0]
root_feature_name = feature_names[root_feature_index]
print("Root splitting feature:", root_feature_name)
print("Root threshold:", root_threshold)
print("======================================================================")
print("Calculate Number of Leaves")
num_leaves = dt_no_scaler.get_n_leaves()
print("Total number of leaves:", num_leaves)

In [ ]:
# Build Random Forest variant with Simple RF
# Define the Random Forest Model
rf = RandomForestClassifier(
 random_state=42,
 n_jobs=-1
)
# Define the search space (Simple RF – no preprocessing)
param_grid = {
 "n_estimators": [50, 100],
 "max_depth": [3, 5, 10]
}
# Define GridSearchCV Details
grid_search_rf = GridSearchCV(
 estimator=rf,
 param_grid=param_grid,
 scoring="roc_auc",
 cv=5,
 n_jobs=-1,
 verbose=1
)
grid_search_rf.fit(X_train, y_train)
print("======================================================================")
print("Random Forest variant with Simple RF")
print("Best parameters:", grid_search_rf.best_params_)
print("Best CV ROC-AUC:", grid_search_rf.best_score_)
best_rf = grid_search_rf.best_estimator_
# Build Random Forest variant with PCA RF
# Define the search space (RF with PCA)
param_grid_pca_rf = {
 "pca__n_components": [10, 20, 30],
 "rf__n_estimators": [50, 100],
 "rf__max_depth": [3, 5, 10]
}
pipeline_rf_pca = Pipeline([
 ("scaler", StandardScaler()),
 ("pca", PCA()),
 ("rf", RandomForestClassifier(
 random_state=42,
 n_jobs=-1
 ))
])
# GridSearchCV setup (5-fold CV, ROC-AUC)
grid_search_pca_rf = GridSearchCV(
 estimator=pipeline_rf_pca,
 param_grid=param_grid_pca_rf,
 scoring="roc_auc",
 cv=5,
 n_jobs=-1,
 verbose=1
)
grid_search_pca_rf.fit(X_train, y_train)
print("======================================================================")
print("Random Forest variant with Simple RF")
print("Best parameters:", grid_search_pca_rf.best_params_)
print("Best CV ROC-AUC:", grid_search_pca_rf.best_score_)
best_pca_rf = grid_search_pca_rf.best_estimator_
# Build Random Forest variant with LLE RF
# Define the search space (RF with LLE)
pipeline_rf_lle = Pipeline([
 ("scaler", StandardScaler()),
 ("lle", LocallyLinearEmbedding(method="standard")),
 ("rf", RandomForestClassifier(
 random_state=42,
 n_jobs=-1
 ))
])
# Define the hyperparameter search space
param_grid_rf_lle = {
 "lle__n_components": [10, 15],
 "lle__n_neighbors": [5, 10, 15],
 "rf__n_estimators": [50, 100],
 "rf__max_depth": [3, 5, 10]
}
# GridSearchCV setup
grid_search_rf_lle = GridSearchCV(
     estimator=pipeline_rf_lle,
 param_grid=param_grid_rf_lle,
 scoring="roc_auc",
 cv=5,
 n_jobs=-1,
 verbose=1
)
grid_search_rf_lle.fit(X_train, y_train)
print("======================================================================")
print("Random Forest variant with LLE RF")
print("Best parameters:", grid_search_rf_lle.best_params_)
print("Best CV ROC-AUC:", grid_search_rf_lle.best_score_)
best_rf_lle = grid_search_rf_lle.best_estimator_

Metrics Report

In [ ]:
# Function to evaluate a fitted GridSearchCV
def evaluate_grid_search(name, grid_search, X_test, y_test):
 best_model = grid_search.best_estimator_
 # Test predictions
 y_test_proba = best_model.predict_proba(X_test)[:, 1]
 y_test_pred = best_model.predict(X_test)
 return {
 "Model": name,
 "Mean ROC-AUC (CV Train)": grid_search.best_score_,
 "ROC-AUC (Test)": roc_auc_score(y_test, y_test_proba),
 "Accuracy (Test)": accuracy_score(y_test, y_test_pred),
 "Training Time (s)": grid_search.refit_time_,
 "Best Hyperparameters": grid_search.best_params_
 }
# Evaluate all three RF variants
results = []
results.append(
 evaluate_grid_search(
 "Simple RF",
      grid_search_rf,
 X_test,
 y_test
 )
)
results.append(
 evaluate_grid_search(
 "RF + PCA",
 grid_search_pca_rf,
 X_test,
 y_test
 )
)
results.append(
 evaluate_grid_search(
 "RF + LLE",
 grid_search_rf_lle,
 X_test,
 y_test
 )
)
# Create final results table
results_df = pd.DataFrame(results)
results_df

Remarks:
Based on the above reaults, the best model is the Simple Random Forest (no
preprocessing). Although RF + LLE achieves the highest mean CV ROC-AUC on the
training data, the Simple RF delivers the best test-set performance with: the best ROCAUC: (0.9934) and the best Accuracy: (0.9561) It requires the shortest time to be trained
as well.
More analytically for the three variants:
- Simple RF: It presents the strongest generalization (best test ROC-AUC and
accuracy), the lowest training time and there is no need for dimensionality reduction
which means no information loss.
- RF + PCA: It presents comparable training ROC-AUC, but noticeable drop in test
performance. Utilizing PCA, removes variance directions that may still be useful for
tree splits. Finally. it is slightly slower than simple RF with no performance gain.
- RF + LLE: It presents highest training ROC-AUC (possible overfitting), worse test
ROC-AUC than both alternatives, it is the slowest to train.